In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'together', 'nbformat', 'rdflib'])
print('Dependencies ready.')

In [ ]:
import json
import os
import random
import re
import hashlib
from collections import Counter
from datetime import datetime
from pathlib import Path

from together import Together
from rdflib import Graph as RDFGraph

print('Imports OK')

In [ ]:
# API key hardcoded for personal development use.
# WARNING: Do not share publicly with key in place.
os.environ['TOGETHER_API_KEY'] = '' # enter your api key

if 'PASTE_YOUR' in os.environ['TOGETHER_API_KEY']:
    raise ValueError('Replace PASTE_YOUR_TOGETHER_KEY_HERE with your actual Together API key.')
print('Together API key set.')

In [ ]:
# -- Model configuration ------------------------------------------------------
GENERATION_MODEL = 'deepseek-ai/DeepSeek-V4-Pro'

# -- Input: Stage 1 v3.1 FULL RUN output (1178 extractable) -------------------
BASE_DIR = Path('/Users/umair/CCO-GRO')
STAGE1_INPUT = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage1_decomposition' / 'stage1_decompositions.json'

# -- Output -------------------------------------------------------------------
OUTPUT_DIR = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage2_extraction'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TURTLE_DIR = OUTPUT_DIR / 'turtle'
TURTLE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# -- Namespaces (Approach B: 3-layer + jurisdiction-separated) ----------------
CCO_NS    = 'https://www.w3id.org/cco/cco#'
GRO_NS    = 'https://w3id.org/cco-gro/onto#'         # Meta-alignment (Layer 4 only)
GRO_UK_NS = 'https://w3id.org/cco-gro/onto/uk#'      # UK domain
GRO_US_NS = 'https://w3id.org/cco-gro/onto/us#'      # USA domain
GRO_CA_NS = 'https://w3id.org/cco-gro/onto/ca#'      # Canada domain
GRO_AU_NS = 'https://w3id.org/cco-gro/onto/au#'      # Australia domain
DATA_NS   = 'https://w3id.org/cco-gro/data#'         # Instance data
PROV_NS   = 'http://www.w3.org/ns/prov#'

# -- Ontology IRIs ------------------------------------------------------------
GRO_ONTOLOGY_IRI    = 'https://w3id.org/cco-gro/onto'
GRO_UK_ONTOLOGY_IRI = 'https://w3id.org/cco-gro/onto/uk'
GRO_US_ONTOLOGY_IRI = 'https://w3id.org/cco-gro/onto/us'
GRO_CA_ONTOLOGY_IRI = 'https://w3id.org/cco-gro/onto/ca'
GRO_AU_ONTOLOGY_IRI = 'https://w3id.org/cco-gro/onto/au'

# -- Jurisdiction -> prefix mapping -------------------------------------------
JURISDICTION_TO_NS_PREFIX = {
    'UK':        'gro-uk',
    'USA':       'gro-us',
    'Canada':    'gro-ca',
    'Australia': 'gro-au',
}
JURISDICTION_TO_ONTOLOGY_IRI = {
    'UK':        GRO_UK_ONTOLOGY_IRI,
    'USA':       GRO_US_ONTOLOGY_IRI,
    'Canada':    GRO_CA_ONTOLOGY_IRI,
    'Australia': GRO_AU_ONTOLOGY_IRI,
}

# -- Runtime ------------------------------------------------------------------
GEN_TEMPERATURE = 0.1
MAX_TOKENS_STAGE2 = 2500
CHECKPOINT_INTERVAL = 25   # Save & log every N chunks

print('Stage 2 Configuration OK')
print(f'  Stage 1 input        : {STAGE1_INPUT}')
print(f'  Output dir           : {OUTPUT_DIR}')
print(f'  Checkpoint interval  : every {CHECKPOINT_INTERVAL} chunks')
print(f'  Jurisdiction prefixes: {JURISDICTION_TO_NS_PREFIX}')

# --  constants -------------------------------------------------------------
RDFS_LABEL_MAX_CHARS = 80
CCO_ROLE_TYPES = {'Role', 'RegulatoryAuthorityRole'}
CCO_AGENT_TYPES = {'Person', 'Organisation', 'Agent', 'RegulatoryAuthorityAgent'}
CCO_RESOURCE_TYPES = {'Resource'}
print(f'v5 constants: RDFS_LABEL_MAX_CHARS={RDFS_LABEL_MAX_CHARS}')
print(f'  Role types  (→ cco:appliesToRole): {sorted(CCO_ROLE_TYPES)}')
print(f'  Agent types (→ gro:hasSubject):    {sorted(CCO_AGENT_TYPES)}')
print(f'  Resource types (→ cco:hasObject):  {sorted(CCO_RESOURCE_TYPES)}')

In [ ]:
# -- Stage 2 LLM Client (V4-Pro with thinking disabled + retry) ------------
import time

MAX_API_RETRIES = 3

_client = None
def get_client():
    global _client
    if _client is None:
        _client = Together()
    return _client

def call_stage2_llm(system: str, user: str, max_tokens: int = MAX_TOKENS_STAGE2, 
                    temperature: float = GEN_TEMPERATURE) -> str:
    """Stage 2: Graph Extraction via DeepSeek-V4-Pro (thinking disabled)."""
    response = get_client().chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        reasoning={"enabled": False},   # CRITICAL: disable thinking for V4-Pro
    )
    return (response.choices[0].message.content or '').strip()

def call_stage2_llm_with_retry(system: str, user: str, max_retries: int = MAX_API_RETRIES,
                                max_tokens: int = MAX_TOKENS_STAGE2, 
                                temperature: float = GEN_TEMPERATURE) -> str:
    """Wrapper with exponential backoff retry for transient API failures."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return call_stage2_llm(system, user, max_tokens, temperature)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait = 2 ** attempt  # 1s, 2s, 4s
                print(f'    API error (attempt {attempt+1}/{max_retries}), retry in {wait}s: {e}')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f'All {max_retries} retries failed: {last_error}')

print('Stage 2 LLM wrapper defined (V4-Pro + thinking disabled + 3-attempt API retry).')

In [ ]:
# -- URI generation (SOTA: source-anchored + 6-char hash) --------------------

def normalize_label(label: str) -> str:
    """Normalize label for consistent hashing."""
    if not label:
        return ''
    return re.sub(r'\s+', ' ', label).strip().lower()


def make_instance_uri(unit_id: str, entity_type: str, label: str) -> str:
    """Generate reproducible instance URI.
    
    Format: data:<unit_id>_<entity_type>_<hash6>
    Hash: sha256(unit_id + entity_type + normalized_label)[:6]
    """
    safe_unit = unit_id.replace('-', '_')
    norm_label = normalize_label(label)
    key = f'{unit_id}|{entity_type}|{norm_label}'
    h = hashlib.sha256(key.encode()).hexdigest()[:6]
    return f'data:{safe_unit}_{entity_type}_{h}'


def make_domain_class_uri(label: str) -> str:
    """Generate domain class URI from descriptive label.
    
    Format: gro:<CamelCaseLabel>
    """
    if not label:
        return 'gro:UnnamedClass'
    # Remove non-alphanumeric, CamelCase the words
    words = re.findall(r'[A-Za-z0-9]+', label)
    camel = ''.join(w.capitalize() for w in words if w)
    return f'gro:{camel}' if camel else 'gro:UnnamedClass'


# Quick tests
print('URI generation tests:')
print(f'  Instance URI: {make_instance_uri("UK-UNIT-00457", "norm", "profit charging prohibition")}')
print(f'  Instance URI: {make_instance_uri("UK-UNIT-00457", "subject", "training provider")}')
print(f'  Domain class: {make_domain_class_uri("Profit Charging Prohibition")}')
print(f'  Domain class: {make_domain_class_uri("Training Provider")}')
print(f'  Domain class: {make_domain_class_uri("Submission Obligation")}')

In [ ]:
# -- CCO whitelist (authoritative — verified against CCO.ttl) ---------------
CCO_CLASSES = {
    'Action', 'Agent', 'Condition', 'Exception', 'Norm', 'Obligation',
    'Organisation', 'Permission', 'Person', 'Prohibition', 'Regulation',
    'RegulatoryAuthorityAgent', 'RegulatoryAuthorityRole', 'Resource',
    'Role', 'RoleHolding',
}
CCO_OBJECT_PROPERTIES = {
    'allocatedTo', 'appliesToRole', 'appliesUnder', 'hasAction',
    'hasCondition', 'hasException', 'hasObject', 'hasRole',
    'holdsRole', 'issues', 'modifiesNorm', 'regulates',
    'specifiesNorm', 'supersedes',
}
CCO_DATA_PROPERTIES = {
    'hasActionExpression', 'hasApplicabilityEnd', 'hasApplicabilityStart',
    'hasConditionExpression', 'hasEndTime', 'hasStartTime',
    'hasValidityEnd', 'hasValidityStart',
}
CCO_FORBIDDEN_TERMS = {
    # Hallucinated classes
    'Document', 'Contract', 'InformationObject', 'InformationResource',
    'State', 'TemporalConstraint', 'FinancialResource', 'GovernmentBody',
    'Quantity', 'Date', 'TimeInterval', 'Event',
    # Hallucinated object properties
    'hasSubject',  # USE gro:hasSubject INSTEAD
    'hasAuthority',
    'hasTemporalConstraint',
    'hasTarget', 'appliesTo', 'concerns',
    # NEW: Hallucinated *Expression data properties (Rule 17)
    'appliesUnderExpression',
    'hasObjectExpression',
    'hasExceptionExpression',
    'hasSubjectExpression',
    'appliesUnderTextual',
    'hasObjectText',
}
GRO_META_PROPERTIES = {'hasSubject'}

PROVISION_TYPE_ROUTING = {
    # === NORM-BEARING TYPES ===
    'Obligation': {
        'cco_norm_class': 'cco:Obligation',
        'create_norm_instance': True,
        'instructions': 'Create cco:Obligation norm instance. Link subject via cco:appliesToRole (Role-type) or gro:hasSubject (Agent-type). Link action via cco:hasAction. Conditions via cco:appliesUnder. Exceptions via cco:hasException.',
    },
    'Permission': {
        'cco_norm_class': 'cco:Permission',
        'create_norm_instance': True,
        'instructions': 'Create cco:Permission norm instance. Same linking pattern as Obligation but represents discretionary right.',
    },
    'Prohibition': {
        'cco_norm_class': 'cco:Prohibition',
        'create_norm_instance': True,
        'instructions': 'Create cco:Prohibition norm instance. The action_or_state represents the FORBIDDEN action. Link via cco:hasAction.',
    },
    'Exception': {
        'cco_norm_class': 'cco:Exception',
        'create_norm_instance': True,
        'instructions': 'Create cco:Exception instance. Conditions on the exception go via cco:hasCondition (Exception is the correct domain for cco:hasCondition). If the modified norm is identifiable, use cco:modifiesNorm.',
    },
    'EligibilityRule': {
        'cco_norm_class': 'cco:Norm',  # generic; deontic_type may refine
        'create_norm_instance': True,
        'requires_condition': True,
        'instructions': 'Create eligibility rule as cco:Norm (or refined to Obligation/Permission/Prohibition if deontic_type is set). MUST include cco:appliesUnder to a Condition instance representing eligibility criteria. Condition instance carries cco:hasConditionExpression with full criteria text.',
    },
    
    # === NON-NORM TYPES (special routing) ===
    'FundingAllocation': {
        'cco_norm_class': None,
        'create_norm_instance': False,
        'create_resource_centric': True,
        'instructions': '''This is a FUNDING ALLOCATION — Resource is primary, not Norm.
GENERATE:
1. A {jurisdiction_prefix}:* domain class for the funding type (subclass of cco:Resource)
2. A data:* instance typed as that class
3. cco:allocatedTo linking Resource → recipient (Role/Agent instance)
4. If monetary_amount is set, store as: data:resource skos:definition "<amount text>" .
5. If wrapping Norm exists ("provider is paid X"), create cco:Permission/Obligation wrapping the allocation.
DO NOT create a freestanding Norm instance unless modal verb is present.''',
    },
    'Definition': {
        'cco_norm_class': None,
        'create_norm_instance': False,
        'create_class_only': True,
        'instructions': '''This is a DEFINITION — generate owl:Class ONLY, NOT a Norm instance.
GENERATE:
1. Prefix declarations
2. ONE {jurisdiction_prefix}:* domain class:
   {jurisdiction_prefix}:DefinedTerm a owl:Class ;
       rdfs:subClassOf <appropriate CCO parent> ;
       rdfs:label "<term name, ≤80 chars>" ;
       skos:definition "<full definition text>" .
3. prov:wasDerivedFrom link to source.
DO NOT create: norm instances, subject instances, action instances, condition instances.
DO NOT use cco:Obligation/Permission/Prohibition in output.''',
    },
    'CalculationRule': {
        'cco_norm_class': None,
        'create_norm_instance': False,
        'create_restriction_pattern': True,
        'instructions': '''This is a CALCULATION RULE — encode as structural definition.
GENERATE:
1. A {jurisdiction_prefix}:* domain class for the calculated entity (subclass of cco:Resource or cco:Condition)
2. Use skos:definition to capture the calculation formula in natural language
3. Conditional logic → cco:hasConditionExpression on a Condition instance
4. DO NOT create deontic Norm instances unless explicit modal verb ("must", "shall", "may not") is in norm_statement.
If no modal verb, this is a structural rule, not a duty.''',
    },
}

def get_provision_type_instructions(provision_type: str, jurisdiction_prefix: str) -> str:
    """Build prompt-injected routing instructions for a provision type."""
    routing = PROVISION_TYPE_ROUTING.get(provision_type)
    if not routing:
        return f"UNKNOWN provision_type '{provision_type}' — treat as generic cco:Norm."
    instr = routing['instructions']
    if '{jurisdiction_prefix}' in instr:
        instr = instr.format(jurisdiction_prefix=jurisdiction_prefix)
    return f"PROVISION TYPE: {provision_type}\nROUTING INSTRUCTIONS:\n{instr}"

print(f'CCO classes        : {len(CCO_CLASSES)}')
print(f'CCO obj properties : {len(CCO_OBJECT_PROPERTIES)}')
print(f'CCO data properties: {len(CCO_DATA_PROPERTIES)}')
print(f'GRO meta props     : {len(GRO_META_PROPERTIES)}')
print(f'Forbidden terms    : {len(CCO_FORBIDDEN_TERMS)}')
print(f'Provision types    : {len(PROVISION_TYPE_ROUTING)}')
for pt in PROVISION_TYPE_ROUTING:
    routing = PROVISION_TYPE_ROUTING[pt]
    norm_class = routing.get('cco_norm_class', 'NONE')
    print(f'  {pt:<20s} -> {norm_class}')

In [ ]:
# -- Stage 2 Prompt v5: provision-type-aware routing -------------------------
STAGE2_SYSTEM = """You are an ontology engineer generating CCO-compliant RDF triples in Turtle syntax.

Your task: given a regulatory provision with Stage 1 decomposition, generate a jurisdiction-aware Turtle graph following the PROVISION TYPE ROUTING INSTRUCTIONS provided.

CRITICAL — CCO whitelist (you MUST follow):
- Only use the 16 CCO classes listed below
- Only use the 14 CCO object properties listed below
- Only use the 8 CCO data properties listed below
- For norm-bearer relations on Agent-types (Person/Organisation/Agent), use gro:hasSubject
- For norm-bearer relations on Role-types, use cco:appliesToRole
- For invented properties, follow the substitution table

Respond with ONLY valid Turtle syntax. No prose, no markdown fences."""

STAGE2_USER_TEMPLATE = """Generate a CCO-compliant Turtle graph for this regulatory provision.

PROVISION ID: {unit_id}
JURISDICTION: {jurisdiction}
JURISDICTION PREFIX: {jurisdiction_prefix}
JURISDICTION NAMESPACE: {jurisdiction_namespace}

PROVISION TEXT:
{text}

STAGE 1 DECOMPOSITION:
{stage1_json}

TOP RETRIEVED ODP PATTERN: {top_odp_label}
STAGE 1 SUGGESTED PRIMARY ODP: {primary_odp}

==============================================================================
PROVISION TYPE ROUTING (CRITICAL — FOLLOW THESE INSTRUCTIONS):
==============================================================================
{routing_instructions}
==============================================================================

CCO WHITELIST — ONLY use terms from these lists
==============================================================================
VALID CCO CLASSES (16 total):
  cco:Action, cco:Agent, cco:Condition, cco:Exception, cco:Norm,
  cco:Obligation, cco:Organisation, cco:Permission, cco:Person,
  cco:Prohibition, cco:Regulation, cco:RegulatoryAuthorityAgent,
  cco:RegulatoryAuthorityRole, cco:Resource, cco:Role, cco:RoleHolding

VALID CCO OBJECT PROPERTIES (14 total):
  cco:allocatedTo, cco:appliesToRole, cco:appliesUnder, cco:hasAction,
  cco:hasCondition, cco:hasException, cco:hasObject, cco:hasRole,
  cco:holdsRole, cco:issues, cco:modifiesNorm, cco:regulates,
  cco:specifiesNorm, cco:supersedes

VALID CCO DATA PROPERTIES (8 total):
  cco:hasActionExpression, cco:hasApplicabilityEnd, cco:hasApplicabilityStart,
  cco:hasConditionExpression, cco:hasEndTime, cco:hasStartTime,
  cco:hasValidityEnd, cco:hasValidityStart

DO NOT USE (common hallucinations):
  cco:hasSubject       -> USE: gro:hasSubject (for Agent-type subjects)
  cco:hasAuthority     -> USE: cco:issues
  cco:hasTemporalConstraint -> USE: cco:hasStartTime / cco:hasEndTime
  cco:Document         -> CREATE: {jurisdiction_prefix}:Document subClassOf cco:Resource
  cco:Contract         -> CREATE: {jurisdiction_prefix}:Contract subClassOf cco:Resource
  cco:State            -> CREATE: {jurisdiction_prefix}:State subClassOf cco:Condition
  cco:FinancialResource -> CREATE: {jurisdiction_prefix}:FinancialResource subClassOf cco:Resource
  cco:GovernmentBody   -> CREATE: {jurisdiction_prefix}:GovernmentBody subClassOf cco:Organisation
  cco:Event            -> CREATE: {jurisdiction_prefix}:Event subClassOf cco:Action

REQUIREMENTS:
1. **Prefixes (REQUIRED — exact URIs)**:
@prefix cco:    <https://www.w3id.org/cco/cco#> .
@prefix gro:    <https://w3id.org/cco-gro/onto#> .
@prefix {jurisdiction_prefix}: <{jurisdiction_namespace}> .
@prefix data:   <https://w3id.org/cco-gro/data#> .
@prefix prov:   <http://www.w3.org/ns/prov#> .
@prefix rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs:   <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:    <http://www.w3.org/2002/07/owl#> .
@prefix skos:   <http://www.w3.org/2004/02/skos/core#> .
@prefix xsd:    <http://www.w3.org/2001/XMLSchema#> .

2. **Norm-to-Subject relation** (when creating Norm instances):
   IF subject.suggested_cco_type ∈ {{Role, RegulatoryAuthorityRole}}:
     → use cco:appliesToRole
   IF subject.suggested_cco_type ∈ {{Person, Organisation, Agent, RegulatoryAuthorityAgent}}:
     → use gro:hasSubject

3. **Domain ontology classes** (use {jurisdiction_prefix}: prefix):
   Each MUST: a owl:Class ; rdfs:subClassOf <CCO parent> ; rdfs:label "<short>" ; skos:definition "<expanded>" .

4. **rdfs:label discipline (Rule 10) — CRITICAL**:
   - rdfs:label MUST be a SHORT IDENTIFIER (2-5 words), NEVER a full sentence.
   - Hard limit: ≤ 80 characters. Aim for ≤ 40 characters.
   - Think of rdfs:label as the entity's "name", not its "description".
   
   -  BAD:  rdfs:label "Permission: Loan amounts for course fees are paid directly to the provider"
   -  GOOD: rdfs:label "Course Fee Loan Payment Permission"
   
   -  BAD:  rdfs:label "Obligation to be an approved provider in order to access the VET program"
   -  GOOD: rdfs:label "Approved Provider Obligation"
   
   -  BAD:  rdfs:label "must continue to meet the course provider requirements after their approval"
   -  GOOD: rdfs:label "Continue Meeting Requirements"
   
   - Long descriptive text goes to:
     • cco:hasActionExpression "..." for Action instances
     • cco:hasConditionExpression "..." for Condition instances
     • skos:definition "..." for Class/Resource/Subject/Object instances

5. **Pronoun preservation (Rule 13)**:
   If subject.resolution_method == 'document_default':
     → Add skos:altLabel "<subject.original_text>" to the subject instance

6. **Property domain discipline (Rule 14) — CRITICAL**:
   CCO property placement rules. Wrong domain = CCO violation.
   
   CORRECT PLACEMENTS:
       Norm → Condition     ⇒ cco:appliesUnder      (on Norm)
       Norm → Action        ⇒ cco:hasAction         (on Norm)
       Norm → Resource      ⇒ cco:hasObject         (on Norm, NOT on Action!)
       Norm → Exception     ⇒ cco:hasException      (on Norm)
       Exception → Condition ⇒ cco:hasCondition     (on Exception only)
       Exception → Norm     ⇒ cco:modifiesNorm      (on Exception)
   
   CRITICAL — cco:hasObject placement:
   - cco:hasObject ALWAYS appears on the Norm instance.
   - cco:hasObject NEVER appears on the Action instance.
   
    BAD:
       data:my_action a cco:Action ;
           cco:hasObject data:my_resource .       # WRONG — Action has no cco:hasObject domain
   
    GOOD:
       data:my_norm a cco:Obligation ;
           cco:hasAction data:my_action ;
           cco:hasObject data:my_resource .       # CORRECT — Norm is the domain of cco:hasObject
       
       data:my_action a cco:Action ;
           rdfs:label "..." ;
           cco:hasActionExpression "..." .         # Action holds only its own expression

7. **PROV provenance (REQUIRED)**:
   Main instance: prov:wasDerivedFrom data:{unit_id_safe}
   data:{unit_id_safe} a prov:Entity ; rdfs:label "Source: {unit_id}" .

8. **xsd:date validity (Rule 16)**:
   - Only emit ^^xsd:date with valid ISO 8601 dates (YYYY-MM-DD)
   - Non-date phrases ("census day", "December 2024") → use cco:hasConditionExpression as string

9. **URIs to use (pre-computed)**:
{precomputed_uris}

10. **Format**:
    - Valid Turtle 1.1
    - No code fences
    - No prose explanation

11. **No invented expression properties (Rule 17) — CRITICAL**:
    - DO NOT invent data properties by appending "Expression" to object properties.
    - The ONLY valid *Expression data properties in CCO are exactly two:
      • cco:hasActionExpression
      • cco:hasConditionExpression
    
    -  FORBIDDEN: cco:appliesUnderExpression
    -  FORBIDDEN: cco:hasObjectExpression  
    -  FORBIDDEN: cco:hasExceptionExpression
    -  FORBIDDEN: cco:hasSubjectExpression
    -  FORBIDDEN: Any other *Expression property not in the whitelist.
    
    - If you need to express a textual qualifier that's not action/condition:
      → Create a cco:Condition instance with cco:hasConditionExpression "..."
      → Link to the Norm via cco:appliesUnder
    
    - Example: For "after their approval" qualifier:
        BAD:  data:my_norm cco:appliesUnderExpression "after their approval" .
        GOOD: data:my_norm cco:appliesUnder data:my_temporal_condition .
               data:my_temporal_condition a cco:Condition ;
                   rdfs:label "Post-approval Period" ;
                   cco:hasConditionExpression "after their approval" .
12. **Universal provenance (Rule 18) — CRITICAL**:
    EVERY data:* instance MUST have prov:wasDerivedFrom data:{unit_id_safe}.
    
    This includes ALL of:
    - The Norm instance
    - The Subject instance
    - The Action instance
    - The Object/Resource instance
    - Any Condition instance
    - Any Exception instance
    - Any Authority instance
    
     BAD (incomplete provenance):
        data:X_norm a cco:Obligation ; ... ; prov:wasDerivedFrom data:X .
        data:X_action a cco:Action ; rdfs:label "..." .   # ← Missing provenance
        data:X_object a gro-uk:Resource ; rdfs:label "..." .   # ← Missing provenance
    
     GOOD (universal provenance):
        data:X_norm a cco:Obligation ; ... ; prov:wasDerivedFrom data:X .
        data:X_action a cco:Action ; rdfs:label "..." ; prov:wasDerivedFrom data:X .
        data:X_object a gro-uk:Resource ; rdfs:label "..." ; prov:wasDerivedFrom data:X .
        data:X_condition a cco:Condition ; rdfs:label "..." ; prov:wasDerivedFrom data:X .

OUTPUT: ONLY Turtle. Start with @prefix, end with last triple."""

print('Stage 2 prompt v5 defined (provision-type routing + 9 rules).')

In [ ]:
# -- Stage 2 Pipeline v5 (with provision-type routing) -----------------------
import re as _re_v5

def _truncate_label(label: str, max_chars: int = None) -> str:
    if max_chars is None:
        max_chars = RDFS_LABEL_MAX_CHARS
    label = (label or '').strip()
    if len(label) <= max_chars:
        return label
    cut = label[:max_chars].rsplit(' ', 1)[0]
    return cut.rstrip(' ,;.:') if cut else label[:max_chars]


def validate_and_fix_turtle(turtle_text: str, stage1_decomp: dict) -> tuple:
    """Post-processor: Rules 10, 13, 14, 14b, 15, 16, 17, 18 deterministically applied."""
    fixes = []
    if not turtle_text:
        return turtle_text, fixes
    
    lines = turtle_text.split('\n')
    out_lines = []
    label_pat = _re_v5.compile(r'rdfs:label\s+"([^"\\]*(?:\\.[^"\\]*)*)"')
    
    full = turtle_text
    type_pat = _re_v5.compile(r'(data:\S+)\s+a\s+([^\s;,.]+)')
    current_type_for_subject = {m.group(1): m.group(2) for m in type_pat.finditer(full)}
    
    active_subject = None
    subject_start_pat = _re_v5.compile(r'^(data:\S+)\s+a\s+([^\s;,.]+)')
    
    # ===== Rule 10: rdfs:label truncation =====
    for idx, line in enumerate(lines):
        m_start = subject_start_pat.match(line)
        if m_start:
            active_subject = m_start.group(1)
        
        m = label_pat.search(line)
        if m:
            label = m.group(1)
            if len(label) > RDFS_LABEL_MAX_CHARS:
                short = _truncate_label(label)
                line = line.replace(f'rdfs:label "{label}"', f'rdfs:label "{short}"')
                
                subj_uri = active_subject or ''
                if '_action_' in subj_uri:
                    expr_prop = 'cco:hasActionExpression'
                elif '_condition_' in subj_uri:
                    expr_prop = 'cco:hasConditionExpression'
                else:
                    expr_prop = 'skos:definition'
                
                if active_subject:
                    out_lines.append(line)
                    indent = _re_v5.match(r'^(\s*)', line).group(1)
                    escaped = label.replace('\\', '\\\\').replace('"', '\\"')
                    out_lines.append(f'{indent}{active_subject} {expr_prop} "{escaped}" .')
                    fixes.append(f'Rule 10: truncated label ({len(label)}->{len(short)}); routed to {expr_prop}')
                    continue
        
        out_lines.append(line)
    
    fixed = '\n'.join(out_lines)
    
    # ===== Rule 14: cco:hasCondition on Norm → cco:appliesUnder =====
    norm_types_lower = ('obligation', 'permission', 'prohibition', 'norm')
    norm_subjects = set()
    for m in type_pat.finditer(fixed):
        uri, typ = m.group(1), m.group(2).lower()
        if any(nt in typ for nt in norm_types_lower) and 'exception' not in typ:
            norm_subjects.add(uri)
    
    if norm_subjects:
        fixed_lines = fixed.split('\n')
        out_lines2 = []
        active_subj = None
        subj_decl_pat = _re_v5.compile(r'^(data:\S+)\s+a\s+')
        for line in fixed_lines:
            d = subj_decl_pat.match(line)
            if d:
                active_subj = d.group(1)
            if 'cco:hasCondition' in line and active_subj in norm_subjects:
                line = line.replace('cco:hasCondition', 'cco:appliesUnder')
                fixes.append(f'Rule 14: rewrote cco:hasCondition -> cco:appliesUnder on {active_subj}')
            out_lines2.append(line)
        fixed = '\n'.join(out_lines2)
    
    # ===== Rule 15: prefix URI normalization =====
    EXPECTED_PREFIX_URIS = {
        'cco':  'https://www.w3id.org/cco/cco#',
        'gro':  'https://w3id.org/cco-gro/onto#',
        'data': 'https://w3id.org/cco-gro/data#',
        'prov': 'http://www.w3.org/ns/prov#',
        'rdf':  'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
        'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
        'owl':  'http://www.w3.org/2002/07/owl#',
        'skos': 'http://www.w3.org/2004/02/skos/core#',
        'xsd':  'http://www.w3.org/2001/XMLSchema#',
    }
    for prefix, canonical in EXPECTED_PREFIX_URIS.items():
        decl_pat = _re_v5.compile(rf'(@prefix {prefix}:\s*)<([^>]+)>')
        m_decl = decl_pat.search(fixed)
        if m_decl and m_decl.group(2) != canonical:
            old_uri = m_decl.group(2)
            fixed = decl_pat.sub(rf'\g<1><{canonical}>', fixed)
            fixes.append(f'Rule 15: normalised @prefix {prefix}: "{old_uri}" -> "{canonical}"')
    
    # ===== Rule 16: strip invalid xsd:date =====
    valid_xsd_date = _re_v5.compile(r'^-?\d{4}-\d{2}-\d{2}(T\d{2}:\d{2}:\d{2}(\.\d+)?(Z|[+-]\d{2}:\d{2})?)?$')
    xsd_date_lit = _re_v5.compile(r'"([^"]*)"\s*\^\^xsd:date(?![A-Za-z])')
    
    def _fix_invalid_date(match):
        val = match.group(1)
        if valid_xsd_date.match(val):
            return match.group(0)
        fixes.append(f'Rule 16: stripped invalid xsd:date type from "{val}"')
        return f'"{val}"'
    
    fixed = xsd_date_lit.sub(_fix_invalid_date, fixed)
    
    # ===== Rule 17: Strip invented *Expression properties =====
    INVENTED_EXPR_PROPS = [
        'cco:appliesUnderExpression',
        'cco:hasObjectExpression',
        'cco:hasExceptionExpression',
        'cco:hasSubjectExpression',
        'cco:hasActionExpressionExpression',
        'cco:appliesUnderTextual',
        'cco:hasObjectText',
    ]
    for invented_prop in INVENTED_EXPR_PROPS:
        if invented_prop in fixed:
            new_lines = []
            stripped_count = 0
            for line in fixed.split('\n'):
                if invented_prop in line:
                    stripped_count += 1
                    continue
                new_lines.append(line)
            if stripped_count > 0:
                fixed = '\n'.join(new_lines)
                fixes.append(f'Rule 17: stripped {stripped_count} line(s) using invented property {invented_prop}')
    
    # ===== Rule 14b: Strip cco:hasObject from Action instances =====
    action_subjects = set()
    action_pat = _re_v5.compile(r'(data:\S+)\s+a\s+cco:Action\b')
    for m in action_pat.finditer(fixed):
        action_subjects.add(m.group(1))
    
    if action_subjects:
        fixed_lines = fixed.split('\n')
        out_lines14b = []
        active_subj14b = None
        subj_decl_pat14b = _re_v5.compile(r'^(data:\S+)\s+a\s+')
        stripped_hasobject = 0
        for line in fixed_lines:
            d = subj_decl_pat14b.match(line)
            if d:
                active_subj14b = d.group(1)
            if 'cco:hasObject' in line and active_subj14b in action_subjects:
                stripped_hasobject += 1
                continue
            out_lines14b.append(line)
        if stripped_hasobject > 0:
            fixed = '\n'.join(out_lines14b)
            fixes.append(f'Rule 14b: stripped {stripped_hasobject} cco:hasObject triple(s) from Action instance(s) (wrong domain)')
    
    # ===== Rule 18: Universal provenance =====
    source_pat = _re_v5.compile(r'(data:\S+)\s+a\s+prov:Entity\b')
    source_match = source_pat.search(fixed)
    if source_match:
        source_uri = source_match.group(1)
        instance_decl_pat = _re_v5.compile(r'^(data:\S+)\s+a\s+', _re_v5.MULTILINE)
        all_instances = set(m.group(1) for m in instance_decl_pat.finditer(fixed))
        all_instances.discard(source_uri)
        
        instances_missing_prov = []
        for instance_uri in all_instances:
            pattern = _re_v5.compile(
                rf'{_re_v5.escape(instance_uri)}\b.*?prov:wasDerivedFrom',
                _re_v5.DOTALL
            )
            if not pattern.search(fixed):
                instances_missing_prov.append(instance_uri)
        
        if instances_missing_prov:
            prov_additions = []
            for inst in instances_missing_prov:
                prov_additions.append(f'{inst} prov:wasDerivedFrom {source_uri} .')
            fixed = fixed.rstrip() + '\n\n# Rule 18: provenance additions\n' + '\n'.join(prov_additions) + '\n'
            fixes.append(f'Rule 18: added prov:wasDerivedFrom to {len(instances_missing_prov)} instance(s) missing provenance')
    
    # ===== Rule 13: skos:altLabel for pronoun-resolved subjects =====
    subj = (stage1_decomp or {}).get('subject') or {}
    if isinstance(subj, dict) and subj.get('resolution_method') == 'document_default':
        original = subj.get('original_text')
        if original and 'skos:altLabel' not in fixed:
            subj_uri_pat = _re_v5.compile(r'(data:\S+_subject_\w+)\s+a\s+')
            m = subj_uri_pat.search(fixed)
            if m:
                subj_uri = m.group(1)
                escaped = original.replace('"', '\\"')
                fixed += f'\n\n{subj_uri} skos:altLabel "{escaped}" .   # Rule 13: pronoun original_text\n'
                fixes.append(f'Rule 13: added skos:altLabel "{original}" on {subj_uri}')
    
    return fixed, fixes

def precompute_uris_for_chunk(stage1_result: dict) -> dict:
    unit_id = stage1_result['unit_id']
    safe_unit = unit_id.replace('-', '_')
    decomp = stage1_result['stage1_decomposition'] or {}
    
    uris = {
        'source_entity': f'data:{safe_unit}',
        'norm_instance': make_instance_uri(unit_id, 'norm', decomp.get('norm_statement', 'norm')),
    }
    subj = decomp.get('subject') or {}
    if subj.get('text'):
        uris['subject_instance'] = make_instance_uri(unit_id, 'subject', subj.get('text'))
    if decomp.get('action_or_state'):
        uris['action_instance'] = make_instance_uri(unit_id, 'action', decomp['action_or_state'])
    if decomp.get('object_or_resource'):
        uris['object_instance'] = make_instance_uri(unit_id, 'object', decomp['object_or_resource'])
    if decomp.get('condition_text'):
        uris['condition_instance'] = make_instance_uri(unit_id, 'condition', decomp['condition_text'])
    if decomp.get('exception_text'):
        uris['exception_instance'] = make_instance_uri(unit_id, 'exception', decomp['exception_text'])
    if decomp.get('authority_text'):
        uris['authority_instance'] = make_instance_uri(unit_id, 'authority', decomp['authority_text'])
    return uris


def format_uris_for_prompt(uris: dict) -> str:
    return '\n'.join(f'  - {role}: {uri}' for role, uri in uris.items())


def run_stage2(stage1_result: dict) -> dict:
    """Run Stage 2 on a single Stage 1 result with provision-type routing."""
    unit_id = stage1_result['unit_id']
    safe_unit = unit_id.replace('-', '_')
    decomp = stage1_result['stage1_decomposition'] or {}
    provision_type = decomp.get('provision_type', 'Unknown')
    primary_odp = decomp.get('likely_primary_odp_label', 'Unknown')
    top_odp_label = stage1_result.get('top_odp_label', 'Unknown')
    
    uris = precompute_uris_for_chunk(stage1_result)
    uri_section = format_uris_for_prompt(uris)
    
    result = {
        'unit_id': unit_id,
        'jurisdiction': stage1_result['jurisdiction'],
        'jurisdiction_prefix': JURISDICTION_TO_NS_PREFIX.get(stage1_result['jurisdiction'], 'gro-uk'),
        'provision_type': provision_type,
        'stage1_decomposition': decomp,
        'precomputed_uris': uris,
        'raw_stage2_response': None,
        'turtle_text': None,
        'parse_status': None,
        'parse_error': None,
        'triple_count': None,
        'v5_fixes_applied': [],
        'stage2_completed_at': None,
    }
    
    try:
        jurisdiction = stage1_result['jurisdiction']
        jur_prefix = JURISDICTION_TO_NS_PREFIX.get(jurisdiction, 'gro-uk')
        jur_namespace = {
            'gro-uk': GRO_UK_NS, 'gro-us': GRO_US_NS,
            'gro-ca': GRO_CA_NS, 'gro-au': GRO_AU_NS,
        }[jur_prefix]
        
        # Build routing instructions for this provision type
        routing_instructions = get_provision_type_instructions(provision_type, jur_prefix)
        
        user_prompt = STAGE2_USER_TEMPLATE.format(
            unit_id=unit_id,
            unit_id_safe=safe_unit,
            jurisdiction=jurisdiction,
            jurisdiction_prefix=jur_prefix,
            jurisdiction_namespace=jur_namespace,
            text=stage1_result['text'],
            stage1_json=json.dumps(decomp, indent=2),
            top_odp_label=top_odp_label,
            primary_odp=primary_odp,
            routing_instructions=routing_instructions,
            precomputed_uris=uri_section,
        )
        
        raw = call_stage2_llm_with_retry(STAGE2_SYSTEM, user_prompt)
        result['raw_stage2_response'] = raw
        
        cleaned = raw.strip()
        if cleaned.startswith('```'):
            lines = [ln for ln in cleaned.splitlines() if not ln.strip().startswith('```')]
            cleaned = '\n'.join(lines).strip()
        
        fixed, fixes = validate_and_fix_turtle(cleaned, decomp)
        result['turtle_text'] = fixed
        result['v5_fixes_applied'] = fixes
        
        g = RDFGraph()
        try:
            g.parse(data=fixed, format='turtle')
            result['parse_status'] = 'parsed'
            result['triple_count'] = len(g)
        except Exception as e:
            result['parse_status'] = 'parse_error'
            result['parse_error'] = str(e)
            result['triple_count'] = 0
    
    except Exception as e:
        result['parse_status'] = 'llm_error'
        result['parse_error'] = f'LLM failure: {e}'
    
    result['stage2_completed_at'] = datetime.now().isoformat()
    return result

print('Stage 2 v5 pipeline defined (provision-type routing integrated).')

In [ ]:
# -- Load Stage 1 v3.1 full run output ---------------------------------------
with open(STAGE1_INPUT) as f:
    stage1_data = json.load(f)

all_results = stage1_data['results']
extractable = [r for r in all_results if r['stage1_status'] == 'extractable']

print(f'Stage 1 total chunks   : {len(all_results)}')
print(f'Extractable for Stage 2: {len(extractable)}')

# Sort for reproducibility
extractable.sort(key=lambda r: r['unit_id'])

# Per-jurisdiction breakdown
by_jur = Counter(r['jurisdiction'] for r in extractable)
print(f'\nPer-jurisdiction (all included):')
for jur, count in sorted(by_jur.items()):
    print(f'  {jur:<10}: {count}')

# Per-provision-type breakdown
by_ptype = Counter(r['stage1_decomposition'].get('provision_type') for r in extractable)
print(f'\nPer-provision-type:')
for pt, count in by_ptype.most_common():
    pct = count / len(extractable) * 100
    print(f'  {str(pt):<25}: {count:4d} ({pct:.1f}%)')

sample = extractable
print(f'\n✓ Total Stage 2 input: {len(sample)} chunks')

In [ ]:
# -- Stage 2 Full Run with Checkpoint Saving --------------------------------
import time

MAX_ATTEMPTS = 2  # 1 initial + 1 retry
stage2_results = []
start_time = time.time()
status_counts = Counter()
fixes_counter = Counter()

print(f'Starting Stage 2 v5 full run: {len(sample)} chunks')
print(f'Checkpoint every {CHECKPOINT_INTERVAL} chunks → {CHECKPOINT_DIR}')
print(f'TTL files → {TURTLE_DIR}')
print('=' * 80)

def save_checkpoint(results, idx):
    """Save intermediate state for monitoring/resumability."""
    checkpoint = {
        'metadata': {
            'completed_at_idx': idx,
            'total_target': len(sample),
            'timestamp': datetime.now().isoformat(),
            'elapsed_min': (time.time() - start_time) / 60,
        },
        'status_counts': dict(status_counts),
        'fixes_counter': dict(fixes_counter),
        'results': results,
    }
    ckpt_path = CHECKPOINT_DIR / f'checkpoint_{idx:04d}.json'
    with open(ckpt_path, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, indent=2, ensure_ascii=False)
    return ckpt_path

for i, s1r in enumerate(sample, 1):
    attempt = 0
    result = None
    while attempt < MAX_ATTEMPTS:
        attempt += 1
        result = run_stage2(s1r)
        if result['parse_status'] == 'parsed':
            break
        if attempt < MAX_ATTEMPTS and result['parse_status'] in ('parse_error', 'llm_error'):
            err = (result.get('parse_error') or 'unknown')[:60]
            print(f'  [{i}/{len(sample)}] {s1r["unit_id"]} RETRY: {err}')
        else:
            break
    
    result['attempts'] = attempt
    stage2_results.append(result)
    status_counts[result['parse_status']] += 1
    for fix in result.get('v5_fixes_applied', []):
        # Extract rule name (e.g., "Rule 10:" → "Rule 10")
        rule = fix.split(':')[0]
        fixes_counter[rule] += 1
    
    # Save TTL file immediately if parsed
    if result['parse_status'] == 'parsed' and result.get('turtle_text'):
        ttl_path = TURTLE_DIR / f'{s1r["unit_id"]}.ttl'
        with open(ttl_path, 'w', encoding='utf-8') as f:
            f.write(result['turtle_text'])
    
    # Per-chunk log
    status = result['parse_status']
    triples = result.get('triple_count', 0) or 0
    ptype = result['provision_type']
    retry = f' (retries={attempt-1})' if attempt > 1 else ''
    print(f'[{i:4d}/{len(sample)}] {s1r["unit_id"]:22s} | {s1r["jurisdiction"]:10s} | {ptype:18s} | {status:12s} | {triples:4d} triples{retry}')
    
    # Checkpoint every N chunks
    if i % CHECKPOINT_INTERVAL == 0:
        ckpt = save_checkpoint(stage2_results, i)
        elapsed = (time.time() - start_time) / 60
        rate = i / elapsed if elapsed > 0 else 0
        eta = (len(sample) - i) / rate if rate > 0 else 0
        print('  ' + '─' * 70)
        print(f'  CHECKPOINT @ {i}: {dict(status_counts)} | {elapsed:.1f} min | rate={rate:.1f}/min | ETA={eta:.0f} min')
        if fixes_counter:
            print(f'  Fixes applied so far: {dict(fixes_counter)}')
        print(f'  Saved: {ckpt.name}')
        print('  ' + '─' * 70)

# Final summary
total_time = (time.time() - start_time) / 60
print('\n' + '=' * 80)
print(f'STAGE 2 v5 FULL RUN COMPLETE')
print('=' * 80)
print(f'Total time: {total_time:.1f} min ({total_time/60:.1f} hr)')
print(f'Total chunks: {len(stage2_results)}')
print(f'\nStatus breakdown:')
for status, count in status_counts.most_common():
    pct = count / len(stage2_results) * 100
    print(f'  {status:15s}: {count:4d} ({pct:.1f}%)')

print(f'\nPost-processor fixes applied:')
for rule, count in fixes_counter.most_common():
    print(f'  {rule:15s}: {count}')

# Save final manifest
manifest = {
    'metadata': {
        'run_type': 'stage2_extraction_full',
        'version': 'v5',
        'stage': 'stage2_graph_extraction',
        'generation_model': GENERATION_MODEL,
        'api_provider': 'Together',
        'stage1_source': str(STAGE1_INPUT),
        'sample_size': len(sample),
        'total_time_min': total_time,
        'created_at': datetime.now().isoformat(),
        'retry_enabled': True,
        'max_attempts_per_chunk': MAX_ATTEMPTS,
        'checkpoint_interval': CHECKPOINT_INTERVAL,
    },
    'status_counts': dict(status_counts),
    'fixes_counter': dict(fixes_counter),
    'results': stage2_results,
}

manifest_path = OUTPUT_DIR / 'stage2_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f'\nFinal manifest: {manifest_path}')
print(f'TTL files: {len(list(TURTLE_DIR.glob("*.ttl")))} files in {TURTLE_DIR}')

In [ ]:
# -- Inspect one Stage 2 result ----------------------------------------------
INSPECT_INDEX = 0  # 0-11

if stage2_results:
    r = stage2_results[INSPECT_INDEX]
    print('=' * 70)
    print(f'UNIT ID    : {r["unit_id"]}')
    print(f'JURISDICT  : {r["jurisdiction"]}')
    print(f'PARSE      : {r["parse_status"]}')
    print(f'TRIPLES    : {r["triple_count"]}')
    if r['parse_error']:
        print(f'ERROR      : {r["parse_error"]}')
    print('=' * 70)
    
    print('\n--- STAGE 1 DECOMPOSITION ---')
    print(json.dumps(r['stage1_decomposition'], indent=2)[:1500])
    
    print('\n--- PRECOMPUTED URIS ---')
    for k, v in r['precomputed_uris'].items():
        print(f'  {k}: {v}')
    
    print('\n--- STAGE 2 TURTLE OUTPUT ---')
    print(r.get('turtle_text') or r.get('raw_stage2_response', '(none)'))
else:
    print('No results to inspect.')

In [ ]:
# -- Quality scan v5.1: CCO whitelist + Rules 10-18 -------------------------
# v5.1 adds: Rules 14b (hasObject on Action), 17 (invented *Expression), 
#            18 (universal provenance)
import re

print('=' * 70)
print('STAGE 2 QUALITY SCAN v5.1 (CCO whitelist + Rules 10-18)')
print('=' * 70)

issues = []
cco_class_pattern = re.compile(r'cco:([A-Z][A-Za-z0-9]+)(?![A-Za-z0-9])')
cco_prop_pattern  = re.compile(r'cco:([a-z][A-Za-z0-9]+)(?![A-Za-z0-9])')

def _has_cco_term(ttl_text: str, term: str) -> bool:
    """Return True iff `cco:term` appears as a complete token (not substring)."""
    return bool(re.search(rf'cco:{re.escape(term)}(?![A-Za-z0-9])', ttl_text))

EXPECTED_PREFIX_URIS = {
    'cco':  'https://www.w3id.org/cco/cco#',
    'gro':  'https://w3id.org/cco-gro/onto#',
    'data': 'https://w3id.org/cco-gro/data#',
    'prov': 'http://www.w3.org/ns/prov#',
    'rdf':  'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
    'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
    'owl':  'http://www.w3.org/2002/07/owl#',
    'skos': 'http://www.w3.org/2004/02/skos/core#',
    'xsd':  'http://www.w3.org/2001/XMLSchema#',
}

VALID_XSD_DATE = re.compile(r'^-?\d{4}-\d{2}-\d{2}(T\d{2}:\d{2}:\d{2}(\.\d+)?(Z|[+-]\d{2}:\d{2})?)?$')
XSD_DATE_LITERAL = re.compile(r'"([^"]*)"\s*\^\^xsd:date(?![A-Za-z])')

INVENTED_EXPR_PROPS = [
    'appliesUnderExpression', 'hasObjectExpression',
    'hasExceptionExpression', 'hasSubjectExpression',
    'appliesUnderTextual', 'hasObjectText',
]

for r in stage2_results:
    flags = []
    uid = r['unit_id']
    
    # Check 1: parse status
    if r['parse_status'] != 'parsed':
        flags.append(f'Parse failed: {r["parse_error"]}')
        issues.append((uid, flags))
        continue
    
    ttl = r['turtle_text'] or ''
    
    # Check 2: required prefixes
    required = ['cco:', 'gro:', 'data:', 'prov:', 'rdfs:', 'owl:']
    missing = [p for p in required if f'@prefix {p}' not in ttl]
    if missing:
        flags.append(f'Missing prefixes: {missing}')
    
    # Check 3: class/typing declarations (owl:Class OR direct CCO typing)
    has_owl_class = bool(re.search(r'a\s+owl:Class', ttl))
    has_direct_cco_typing = bool(re.search(r'data:\S+\s+a\s+cco:[A-Z]', ttl))
    if not has_owl_class and not has_direct_cco_typing:
        flags.append('No class declarations or direct CCO typing')
    
    # Check 4: subClassOf relations OR direct CCO typing
    if 'rdfs:subClassOf cco:' not in ttl and not has_direct_cco_typing:
        flags.append('No subclass relations and no direct CCO typing')
    
    # Check 5: provenance link
    if 'prov:wasDerivedFrom' not in ttl:
        flags.append('Missing prov:wasDerivedFrom')
    
    # Check 6: instance typing (skip for Definitions - they only declare classes)
    provision_type = r.get('provision_type', '')
    if provision_type != 'Definition':
        instance_pattern = re.compile(r'data:\S+\s+a\s+(cco:|gro)', re.MULTILINE)
        if not instance_pattern.search(ttl):
            flags.append('No instances typed as cco:* or gro*:*')
    
    # Check 7: rdfs:label
    if 'rdfs:label' not in ttl:
        flags.append('No rdfs:label annotations')
    
    # Check 8: jurisdiction prefix correctness
    expected_prefix = r['jurisdiction_prefix']
    other_prefixes = [p for p in ['gro-uk', 'gro-us', 'gro-ca', 'gro-au'] if p != expected_prefix]
    if f'@prefix {expected_prefix}:' not in ttl:
        flags.append(f'Expected jurisdiction prefix {expected_prefix}: missing')
    for other in other_prefixes:
        if f'@prefix {other}:' in ttl or f' {other}:' in ttl:
            flags.append(f'Cross-jurisdiction leakage: {other}:')
    
    # Check 9: CCO class whitelist
    classes_used = set(cco_class_pattern.findall(ttl))
    invalid_classes = classes_used - CCO_CLASSES
    if invalid_classes:
        flags.append(f'Invalid CCO classes: {sorted(invalid_classes)}')
    
    # Check 10: CCO property whitelist
    props_used = set(cco_prop_pattern.findall(ttl))
    invalid_props = props_used - CCO_OBJECT_PROPERTIES - CCO_DATA_PROPERTIES
    if invalid_props:
        flags.append(f'Invalid CCO properties: {sorted(invalid_props)}')
    
    # Check 11: forbidden term usage
    found_forbidden = [term for term in CCO_FORBIDDEN_TERMS if _has_cco_term(ttl, term)]
    if found_forbidden:
        flags.append(f'Forbidden cco:* terms used: {found_forbidden}')
    
    # Check 12: gro:hasSubject usage encouraged
    if 'gro:hasSubject' not in ttl and _has_cco_term(ttl, 'hasSubject'):
        flags.append('Used cco:hasSubject (invalid) instead of gro:hasSubject')
    
    # ===== v4/v5 rule checks =====
    
    label_pat_v4 = re.compile(r'rdfs:label\s+"([^"]*)"')
    
    # Rule 10: rdfs:label length
    for L in label_pat_v4.findall(ttl):
        if len(L) > RDFS_LABEL_MAX_CHARS:
            flags.append(f'Rule 10 violation: rdfs:label > {RDFS_LABEL_MAX_CHARS} chars ({len(L)}) — "{L[:60]}..."')
            break
    
    # Rule 11: subject-role hookup discipline
    s1_subj = (r.get('stage1_decomposition') or {}).get('subject') or {}
    cco_type = s1_subj.get('suggested_cco_type', '')
    used_appliesToRole = _has_cco_term(ttl, 'appliesToRole')
    if cco_type in CCO_ROLE_TYPES:
        if not used_appliesToRole:
            flags.append(f'Rule 11 violation: subject is {cco_type} (Role-type) but cco:appliesToRole not used')
        if 'gro:hasSubject' in ttl:
            flags.append(f'Rule 11 violation: subject is {cco_type} (Role-type) but gro:hasSubject also used')
    elif cco_type in CCO_AGENT_TYPES:
        if 'gro:hasSubject' not in ttl:
            flags.append(f'Rule 11 violation: subject is {cco_type} (Agent-type) but gro:hasSubject not used')
        if used_appliesToRole:
            flags.append(f'Rule 11 violation: subject is {cco_type} (Agent-type) but cco:appliesToRole also used')
    
    # Rule 12: subject fidelity
    s1_subj_text = (s1_subj.get('text') or '').strip().lower()
    if s1_subj_text:
        ttl_labels_lower = [L.lower() for L in label_pat_v4.findall(ttl)]
        matched = any(s1_subj_text in L or L in s1_subj_text for L in ttl_labels_lower)
        if not matched:
            flags.append(f'Rule 12 violation: Stage 1 subject "{s1_subj_text[:60]}" not found in any rdfs:label')
    
    # Rule 13: pronoun audit trail
    if s1_subj.get('resolution_method') == 'document_default':
        original = s1_subj.get('original_text', '')
        if 'skos:altLabel' not in ttl:
            flags.append(f'Rule 13 violation: subject resolved from pronoun "{original}" but no skos:altLabel triple')
    
    # Rule 14: cco:hasCondition domain discipline (on Norm-type subjects)
    norm_types_lower = ('obligation','permission','prohibition','norm')
    type_decl_pat = re.compile(r'(data:\S+)\s+a\s+([^\s;,]+)')
    norm_subjects = set()
    for tm in type_decl_pat.finditer(ttl):
        uri, typ = tm.group(1), tm.group(2).lower()
        if any(nt in typ for nt in norm_types_lower) and 'exception' not in typ:
            norm_subjects.add(uri)
    hascond_word_pat = re.compile(r'cco:hasCondition(?![A-Za-z])')
    active_s = None
    subj_decl_pat = re.compile(r'^(data:\S+)\s+a\s+')
    rule14_violations = 0
    for ln in ttl.split('\n'):
        dm = subj_decl_pat.match(ln)
        if dm:
            active_s = dm.group(1)
        if hascond_word_pat.search(ln) and active_s in norm_subjects:
            rule14_violations += 1
    if rule14_violations:
        flags.append(f'Rule 14 violation: {rule14_violations} cco:hasCondition triple(s) on Norm-type subject (use cco:appliesUnder)')
    
    # Rule 14b: cco:hasObject on Action instances (wrong domain)
    action_subjects_scan = set()
    for m in re.finditer(r'(data:\S+)\s+a\s+cco:Action\b', ttl):
        action_subjects_scan.add(m.group(1))
    if action_subjects_scan:
        active_s = None
        rule14b_violations = 0
        for ln in ttl.split('\n'):
            dm = subj_decl_pat.match(ln)
            if dm:
                active_s = dm.group(1)
            if 'cco:hasObject' in ln and active_s in action_subjects_scan:
                rule14b_violations += 1
        if rule14b_violations:
            flags.append(f'Rule 14b violation: {rule14b_violations} cco:hasObject triple(s) on Action subject (wrong domain)')
    
    # Rule 15: prefix URI integrity
    prefix_mismatches = []
    for prefix, exp_uri in EXPECTED_PREFIX_URIS.items():
        decl = re.search(rf'@prefix {prefix}:\s*<([^>]+)>', ttl)
        if decl and decl.group(1) != exp_uri:
            prefix_mismatches.append(f'{prefix}: got "{decl.group(1)}"')
    if prefix_mismatches:
        flags.append(f'Rule 15 violation: prefix URI mismatches: {prefix_mismatches}')
    
    # Rule 16: xsd:date literal validity
    invalid_dates = []
    for d in XSD_DATE_LITERAL.findall(ttl):
        if not VALID_XSD_DATE.match(d):
            invalid_dates.append(d)
    if invalid_dates:
        flags.append(f'Rule 16 violation: invalid xsd:date literal(s): {invalid_dates}')
    
    # Rule 17: invented *Expression properties
    invented_exprs = []
    for invented in INVENTED_EXPR_PROPS:
        if _has_cco_term(ttl, invented):
            invented_exprs.append(invented)
    if invented_exprs:
        flags.append(f'Rule 17 violation: invented properties used: {invented_exprs}')
    
    # Rule 18: universal provenance
    source_match = re.search(r'(data:\S+)\s+a\s+prov:Entity\b', ttl)
    if source_match:
        instance_decls = set(re.findall(r'^(data:\S+)\s+a\s+', ttl, re.MULTILINE))
        instance_decls.discard(source_match.group(1))
        missing_prov = []
        for inst in instance_decls:
            inst_block_pat = re.compile(
                rf'{re.escape(inst)}\b.*?prov:wasDerivedFrom',
                re.DOTALL
            )
            if not inst_block_pat.search(ttl):
                missing_prov.append(inst)
        if missing_prov:
            flags.append(f'Rule 18 violation: {len(missing_prov)} instance(s) missing prov:wasDerivedFrom')
    
    # Final: append flags if any
    if flags:
        issues.append((uid, flags))

# Print results
if not issues:
    print('\n✓ No quality issues detected across all results.')
else:
    for uid, flags in issues:
        print(f'\n{uid}:')
        for f in flags:
            print(f'  - {f}')

print(f'\n--- Summary ---')
print(f'Total results       : {len(stage2_results)}')
print(f'Results with issues : {len(issues)}')
print(f'Results clean       : {len(stage2_results) - len(issues)}')

In [ ]:
# # Extract TTL files from existing manifest (no LLM rerun needed) temperory.
# import json
# from pathlib import Path

# manifest_path = OUTPUT_DIR / 'stage2_manifest.json'
# with open(manifest_path, encoding='utf-8') as f:
#     manifest = json.load(f)

# ttl_saved = 0
# ttl_skipped = 0
# for r in manifest['results']:
#     if r['parse_status'] == 'parsed' and r.get('turtle_text'):
#         ttl_path = TURTLE_DIR / f'{r["unit_id"]}.ttl'
#         with open(ttl_path, 'w', encoding='utf-8') as f:
#             f.write(r['turtle_text'])
#         ttl_saved += 1
#     else:
#         ttl_skipped += 1

# print(f'TTL files written:  {ttl_saved}')
# print(f'TTL files skipped:  {ttl_skipped}')
# print(f'Output directory:   {TURTLE_DIR}')

# # Verify
# ttls_on_disk = sorted(TURTLE_DIR.glob('*.ttl'))
# print(f'\nVerification: {len(ttls_on_disk)} .ttl files on disk')

In [ ]:
# # Check how many "no_instances" are actually Definitions (correct routing)
# definition_with_no_instances = 0
# non_definition_with_no_instances = 0

# # Re-categorize critical chunks
# for uid in critical_chunks:
#     for r in stage2_results:
#         if r['unit_id'] == uid:
#             ptype = r.get('provision_type', 'Unknown')
#             ttl = r.get('turtle_text', '')
#             has_instances = bool(re.search(r'data:\S+\s+a\s+(cco:|gro)', ttl, re.MULTILINE))
            
#             if not has_instances:
#                 if ptype == 'Definition':
#                     definition_with_no_instances += 1
#                 else:
#                     non_definition_with_no_instances += 1
#             break

# print(f"Definitions with no instances (CORRECT): {definition_with_no_instances}")
# print(f"Non-Definitions with no instances (PROBLEM): {non_definition_with_no_instances}")
# print(f"\n=== Provision types in critical_chunks ===")
# ptype_in_critical = Counter()
# for uid in critical_chunks:
#     for r in stage2_results:
#         if r['unit_id'] == uid:
#             ptype_in_critical[r.get('provision_type', 'Unknown')] += 1
#             break
# for pt, c in ptype_in_critical.most_common():
#     print(f"  {pt}: {c}")